# Tahap 3 — Tarik Data Cuaca Riil dari NASA POWER

Notebook ini menarik data cuaca **bulanan riil** (2023–2025) untuk 3 kategori
regional Sulsel (Hydro, Solar, Wind) dari NASA POWER API — menggantikan
fitur Cuaca proxy statis yang sebelumnya diulang setiap tahun.

**Koordinat (hasil Tahap 2, Opsi A):**
- **Hydro** — centroid tertimbang kapasitas 7 PLTA besar (Luwu Timur/Luwu)
- **Solar** — titik representatif provinsi (Makassar/Maros), simplifikasi
  karena PLTS+PLTS Atap tersebar di 45+ lokasi kecil
- **Wind** — centroid tertimbang PLTB Sidrap + PLTB Jeneponto

Jalankan sel-sel di bawah secara berurutan. Butuh koneksi internet aktif.

In [ ]:
!pip install requests pandas -q

## 1. Konfigurasi

In [ ]:
import requests
import pandas as pd

# Konfigurasi 3 titik koordinat kategori (hasil Tahap 2, Opsi A)
LOKASI = {
    "Hydro": {"lat": -2.86, "lon": 120.79, "parameter": "PRECTOTCORR"},
    "Solar": {"lat": -5.10, "lon": 119.60, "parameter": "ALLSKY_SFC_SW_DWN"},
    "Wind":  {"lat": -4.64, "lon": 119.82, "parameter": "WS10M"},
}

START_YEAR = 2023
END_YEAR = 2025  # ganti ke 2026 kalau butuh data untuk validasi forecast

BASE_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

print("Konfigurasi siap:")
for kategori, info in LOKASI.items():
    print(f"  {kategori}: lat={info['lat']}, lon={info['lon']}, parameter={info['parameter']}")

## 2. Fungsi penarik data

In [ ]:
def tarik_data(nama_kategori: str, lat: float, lon: float, parameter: str) -> list:
    """Tarik satu parameter cuaca bulanan untuk satu titik koordinat."""
    params = {
        "parameters": parameter,
        "community": "RE",  # Renewable Energy community
        "longitude": lon,
        "latitude": lat,
        "format": "JSON",
        "start": START_YEAR,
        "end": END_YEAR,
    }

    print(f"  Menarik {nama_kategori} ({parameter}) di ({lat}, {lon})...")
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    try:
        monthly = data["properties"]["parameter"][parameter]
    except KeyError:
        raise RuntimeError(
            f"Struktur respons tidak sesuai ekspektasi untuk {nama_kategori}. "
            f"Cek manual: {resp.url}"
        )

    rows = []
    for key, value in monthly.items():
        # Format key: "YYYYMM", contoh "202301".
        # Bulan "13" = rata-rata tahunan, bukan bulan sungguhan -> dilewati.
        tahun, bulan = key[:4], key[4:6]
        if bulan == "13":
            continue
        if value == -999.0:  # kode missing value standar NASA POWER
            print(f"    [PERINGATAN] Data hilang untuk {tahun}-{bulan}, dilewati")
            continue
        rows.append({
            "Tahun": int(tahun),
            "Bulan": int(bulan),
            "Kategori": nama_kategori,
            "Parameter_Cuaca": parameter,
            "Nilai_Cuaca": value,
            "Lat": lat,
            "Lon": lon,
        })
    return rows

## 3. Eksekusi — tarik data untuk 3 kategori

In [ ]:
semua_data = []
print(f"Menarik data cuaca NASA POWER, {START_YEAR}-{END_YEAR}...\n")

for kategori, info in LOKASI.items():
    hasil = tarik_data(kategori, info["lat"], info["lon"], info["parameter"])
    semua_data.extend(hasil)
    print(f"    -> {len(hasil)} baris berhasil ditarik\n")

df = pd.DataFrame(semua_data)
df = df.sort_values(["Kategori", "Tahun", "Bulan"]).reset_index(drop=True)
print(f"Total baris ditarik: {len(df)}")

## 4. Validasi

Harus 36 baris per kategori (12 bulan × 3 tahun 2023–2025) kalau semua berhasil.

In [ ]:
print("Jumlah baris per kategori:")
print(df.groupby("Kategori").size())

print("\nCek missing value per kategori:")
for kategori in LOKASI:
    jumlah = len(df[df["Kategori"] == kategori])
    status = "OK" if jumlah == 36 else f"KURANG ({jumlah}/36) - cek peringatan di atas"
    print(f"  {kategori}: {status}")

## 5. Pratinjau & simpan ke CSV

In [ ]:
df.head(15)

In [ ]:
out_path = "cuaca_riil_regional.csv"
df.to_csv(out_path, index=False)
print(f"Disimpan ke {out_path}")

# Kalau di Google Colab, unduh otomatis:
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    print("(Bukan lingkungan Colab — file tersimpan di direktori kerja lokal)")

## Langkah berikutnya

Setelah `cuaca_riil_regional.csv` berhasil dibuat, ini menggantikan kolom
Cuaca proxy statis di dataset regional. Lanjut ke **Tahap 4**: disagregasi
tahunan → bulanan memakai data cuaca riil ini, dengan logika berbeda per
kategori:
- **Solar & Wind**: proporsi langsung (`produksi_bulan = produksi_tahunan × cuaca_bulan / total_cuaca_tahun`)
- **Hydro**: rata-rata bergerak 2–3 bulan (bukan curah hujan instan) untuk
  mendekati efek tampungan waduk